# 📘 Colab Notebook: Open AI Response/Output Evaluation Using Llumo

## 📝 Notebook Overview
This notebook helps you evaluate Output queries using Llumo’s powerful input-level metrics to ensure quality and safety:

### ✨ Metrics included:
  
- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
- ▶ Hallucination
  
---

## 🚀 What you will do in this notebook:
- 📂 Load user queries, get context according to the query, and get the output from OpenAI.  
- 🤖 Evaluate the queries for bias, correctness, completeness, and harmfulness  
- 📊 View the detailed evaluation results  
---


 ### **⚙️Install Dependencies**

In [1]:
!pip install llumo -q
!pip install openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 3.2 MB/s eta 0:00:00


### **Import Required Libraries**

In [5]:
import pandas as pd
import getpass
import os
import requests
import requests
from openai import OpenAI


### **🔑 Setup OpenAI API Key & Llumo API key**

In [6]:
# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

### **Function to get the context from Database**

In [30]:

# ✅ Function to get context for a single query
def get_context(query):
    url = "https://llumo.ai/function/get-context-from-db"
    reqBody = {"query": query}
    response = requests.post(url, json=reqBody)
    return response.json()["contexts"]


### **🧾 Step 5: Generating the Outputs**




The data used for evaluation will be in the following Example format:

```
[  
  {
    "query": "What is the capital of France?",
    "context": "France is a country in Europe.Its capital city is Paris.",
    "output": "The capital of France is Paris."
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "context": "'Romeo and Juliet' is a tragedy by William Shakespeare.It is about two lovers from feuding families.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."
  }
]
```



In [36]:

# ✅ Sample input data with only queries
data = [
    {"query": "What do I do if my product is missing parts?"},
    {"query": "How can GreenWave Energy help my business reduce its carbon footprint?"},
    {"query": "What is the process for returning a laptop if it’s defective?"}
]

# ✅ OpenAI API initialization
client = OpenAI(api_key=openai_key)

# ✅ Final result list - Input Eval Data
results = []

# ✅ Fetch context & generate output, append to results
for item in data:
    query = item["query"]

    # Step 1: Fetch context
    context = get_context([query])[0]

    # Step 2: Generate LLM output
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {
                "role": "user",
                "content": f"Give answer to the given query: {query}, using the given context: {context}."
            }
        ],
        temperature=0.7
    )
    output = response.choices[0].message.content

    # Step 3: Append full result
    results.append({
        "query": query,
        "context": context,
        "output": output
    })



### 📄 **Input Data with keys — "query", "context", "output"**
Preview the enriched data that will be passed for output evaluation.


In [37]:
results[0]

{'query': 'What do I do if my product is missing parts?',
 'context': 'TechFuture is where innovation meets quality. Our product selection includes the latest smartphones, laptops, tablets, and smart home gadgets. We offer free standard shipping on orders over $100 and a one-year warranty on all items. Our return policy allows for returns within 30 days if the product is in its original, unopened packaging. For assistance with returns, warranty claims, or any product-related inquiries, our customer support team is available 24\\/7. At TechFuture, we are dedicated to providing top-notch products and service to enhance your tech experience.',
 'output': 'If your product from TechFuture is missing parts, you should reach out to our customer support team who are available 24/7. They will assist you with any product-related inquiries including missing parts. You can also make a warranty claim since all items come with a one-year warranty.'}

### 🤖 **Step 5: Initialize Llumo Client And Evaluate Output**
This block initializes the `LlumoClient` and evaluates the quality and safety of output using selected KPIs like:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness

Additional Metrics:
- Context Utilization
- Hallucination
  


In [38]:

# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key = llumo_key)  # Replace with actual API key

evalDf = client.evaluateMultiple(
    data = results,  # Input Data
    evals = ["Response Completeness", "Response Correctness", "Response Bias","Context Utilization" ,"Hallucination"],  # Selected evaluation KPIs
    prompt_template = "Give answer to the given query: {{query}}, using the given context: {{context}}.",  # Prompt used for generation
    createExperiment = False,   # Set to True to save results as an experiment on the Llumo platform. If False, returns results as a DataFrame or A Python Dict. - Optional
    getDataFrame = True, # Return result as a DataFrame (True) or dictionary (False) - Optional
    )



Processing Batches: 100%|██████████| 5/5 [00:18<00:00,  3.64s/batch]


###**📊 View Evaluation Result Table**

In [39]:
evalDf

,query,context,output,Response Completeness,Response Completeness Reason,Response Correctness,Response Correctness Reason,Response Bias,Response Bias Reason,Context Utilization,Context Utilization Reason,Hallucination,Hallucination Reason
0,What do I do if my product is missing parts?,TechFuture is where innovation meets quality. ...,If your product from TechFuture is missing par...,88,The response directly answers the query and in...,100,The response accurately addresses the query ab...,1,"The response does not contain any biased, unfa...",86,The response effectively uses the context to a...,37,The output includes minor details not explicit...
1,How can GreenWave Energy help my business redu...,GreenWave Energy is a leading provider of rene...,GreenWave Energy can help your business reduce...,99,The response directly answers the query and in...,99,The response accurately and completely answers...,2,The response does not express or reinforce bia...,99,The response perfectly utilizes all the releva...,20,The output is almost fully supported by the co...
2,What is the process for returning a laptop if ...,ElectroCenter is where we offer the latest in ...,"If your laptop is defective, you can return it...",99,The response accurately and fully answers the ...,99,The response accurately reflects the return po...,2,The response provides information about return...,94,The response effectively uses the context to p...,37,The output includes minor details not explicit...
